<a href="https://colab.research.google.com/github/yasaswini1408/Prompt_Engineering/blob/main/Experiment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain-google-genai langchain_core pydantic PyMuPDF


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 28.0 MB/s eta 0:00:00
  Attempting uninstall: langchain_core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [2]:
import os
import json
from typing import List
import fitz
# PyMuPDF
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

In [4]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.0)
# Define the structural blueprint (Omitting Age/Gender for safety compliance)
class ResumeSchema(BaseModel):
  candidate_name: str = Field(
      description="The full name of the applicant. Default to 'UNKNOWN' "
      "if missing."     )
  phone_number: str = Field(description="The contact phone number.")
  email: str = Field(description="The candidate's contact email address.")
  total_years_of_experience: float = Field(
      description="Total years of professional experience as a float, "
      "consider present date as June 2026."     )
  list_of_skills: List[str] = Field(
      description="List of key technical skills, frameworks, or "
      "methodologies."     )
  is_good_fit: bool = Field(
      description="True if the candidate has relevant technical skills or "
      "matches professional standards and job description, "
      "otherwise False."     )
  reason: str = Field(
      description="Give reasoning why the candidate is good fit or bad fit "
      "based on resume and job description"     )
  summary: str = Field(
      description="A concise summary of the candidate's career history, "
      "education and projects."     )
  highest_education: str = Field(
      description="The highest academic degree attained (e.g., B.Tech, "
      "MS, PhD)."     )
  # Instantiate the Json Output Parser bound to our Pydantic schema
output_parser = JsonOutputParser(pydantic_object=ResumeSchema)

In [11]:
screener_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an advanced HR Data Extraction pipeline. Extract "
     "structural attributes from the raw resume text.\n\n"
     "{format_instructions}"),
      ("human", "Raw Resume Document Text:\n{resume_text}, Job Description:") ])
# Pre-fill the static schema instructions into the prompt early
# This converts our 2-variable prompt into a clean 1-variable prompt
screener_prompt = screener_prompt.partial(
    format_instructions=output_parser.get_format_instructions() )
# Assemble the streamlined chain
extraction_chain = screener_prompt | llm | output_parser
# ===================================================================== # Step 5: Runtime Execution & Text Extraction # ===================================================================== # Take user input for the file path
sample_pdf_path = input("Enter the path to the resume PDF: ")
job_description = input("Enter the job description: ")
# Inline PyMuPDF extraction loop
raw_document_text = ""
with fitz.open(sample_pdf_path) as doc:
  for page in doc:
    raw_document_text += page.get_text()
print("\n--- Executing Structured Output Chain (Template Pattern) ---") # Execute the chain: Thanks to .partial(), we pass dynamic text variables
parsed_json_dict = extraction_chain.invoke({ "resume_text": raw_document_text,
                                            "job_description": job_description }) # ===================================================================== # Step 6: Output Display # =====================================================================
print("\n--- Verified JSON Dictionary Output ---")
# Print the native dictionary formatted beautifully using python's built-in tool
print(json.dumps(parsed_json_dict, indent=2))

Enter the path to the resume PDF: /content/sample_resume.pdf
Enter the job description: Data Engineer with experience in building scalable ETL pipelines, real-time data streaming systems, and cloud-based analytics platforms using Python, SQL, Spark, Kafka, Airflow, and AWS/GCP. Skilled in data warehousing, workflow automation, big data processing, and collaborating with cross-functional teams to deliver reliable, highperformance data solutions for analytics and machine learning applications.

--- Executing Structured Output Chain (Template Pattern) ---

--- Verified JSON Dictionary Output ---
{
  "candidate_name": "JOHN DOE",
  "phone_number": "+1-555-789-4561",
  "email": "johndoe@gmail.com",
  "total_years_of_experience": 7.0,
  "list_of_skills": [
    "Python",
    "SQL",
    "Scala",
    "ETL/ELT Pipelines",
    "Data Modeling",
    "Data Warehousing",
    "Apache Spark",
    "Hadoop",
    "Kafka",
    "Hive",
    "PostgreSQL",
    "MySQL",
    "MongoDB",
    "Snowflake",
    "BigQ